# 01 Baseline Training

In [ ]:
import os
from PIL import Image
import numpy as np
import tensorflow as tf
from functools import partial
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

In [ ]:
low_light_dir = "../data/train/low"
enhanced_dir = "../data/train/high"

def load_image(image_path, size=(256, 256)):
    img = Image.open(image_path).convert('RGB')
    img = img.resize(size)
    img = np.array(img).astype(np.float32) / 255.0
    return img

low_light_images, enhanced_images = [], []
for img_name in sorted(os.listdir(low_light_dir)):
    low_img_path = os.path.join(low_light_dir, img_name)
    enhanced_img_path = os.path.join(enhanced_dir, img_name)
    if os.path.exists(enhanced_img_path):
        low_light_images.append(load_image(low_img_path))
        enhanced_images.append(load_image(enhanced_img_path))

low_light_array = np.array(low_light_images).astype(np.float32)
enhanced_array = np.array(enhanced_images).astype(np.float32)
print("Dataset Loaded. Shape of low-light images:", low_light_array.shape)

In [ ]:
def unet_model(input_shape=(256, 256, 3)):
    inputs = tf.keras.layers.Input(input_shape)
    conv1 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(inputs)
    pool1 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1)
    conv2 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(pool1)
    pool2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(conv2)
    bottleneck = tf.keras.layers.Conv2D(256, 3, activation='relu', padding='same')(pool2)
    upconv2 = tf.keras.layers.Conv2DTranspose(128, 2, strides=2, padding='same')(bottleneck)
    concat2 = tf.keras.layers.concatenate([conv2, upconv2], axis=-1)
    conv3 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(concat2)
    upconv1 = tf.keras.layers.Conv2DTranspose(64, 2, strides=2, padding='same')(conv3)
    concat1 = tf.keras.layers.concatenate([conv1, upconv1], axis=-1)
    conv4 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(concat1)
    outputs = tf.keras.layers.Conv2D(3, 1, activation='sigmoid')(conv4)
    return tf.keras.Model(inputs=[inputs], outputs=[outputs])

model = unet_model()
model.summary()

In [ ]:
from tensorflow.image import psnr, ssim

def psnr_metric(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
    return tf.reduce_mean(psnr(y_true, y_pred, max_val=1.0))

def ssim_metric(y_true, y_pred):
    y_true = tf.clip_by_value(y_true, 0.0, 1.0)
    y_pred = tf.clip_by_value(y_pred, 0.0, 1.0)
    return tf.reduce_mean(ssim(y_true, y_pred, max_val=1.0))

def color_alignment_loss(y_true, y_pred):
    mean_true = tf.reduce_mean(y_true, axis=[1,2])
    mean_pred = tf.reduce_mean(y_pred, axis=[1,2])
    std_true = tf.math.reduce_std(y_true, axis=[1,2])
    std_pred = tf.math.reduce_std(y_pred, axis=[1,2])
    return tf.reduce_mean(tf.square(mean_true - mean_pred)) + tf.reduce_mean(tf.square(std_true - std_pred))

def combined_loss(y_true, y_pred):
    mse_loss = tf.reduce_mean(tf.square(y_true - y_pred))
    cal_loss = color_alignment_loss(y_true, y_pred)
    return mse_loss + 0.01 * cal_loss

model.compile(optimizer='adam', loss=combined_loss, metrics=['mse', psnr_metric, ssim_metric])

In [ ]:
history = model.fit(low_light_array, enhanced_array, epochs=100, batch_size=4)
os.makedirs("../models", exist_ok=True)
model.save("../models/original_baseline_model.keras")